# Ariel 2025 — Notebook 2a: Build Physics Features (CPU)

Chạy **một lần trên CPU** để trích physics-based features từ raw light curves (cho nhóm **ML-based**), rồi push lên branch `running`. `train_ml.ipynb` sẽ load feature trực tiếp thay vì build lại.

> Tiền xử lý: ADC calibrate → bad-pixel/dark/flat → CDS → temporal binning → light curves → transit detection → normalize/detrend → **physics features** (depth / multi-scale / shape / noise / stellar).
> Cùng `PreprocessConfig` với `prepare_sequence.ipynb` nên nhất quán về calibration; chỉ khác bước feature.

In [ ]:
# === Setup: clone running branch ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src", "src", "../src"]:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using src from:", _p); break
else:
    print("WARNING: src not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Cấu hình
Giữ nguyên `LIMIT` / `TIME_BINS` khi chạy `train_ml.ipynb` để load đúng file.

In [ ]:
from config import PreprocessConfig, FeatureConfig, DatasetConfig
from data_io import ArielDataRepository
from pipeline import ArielPreprocessFeaturePipeline
from dataset_builder import ArielDatasetBuilder

LIMIT = None       # None = full (khuyến nghị); phải khớp khi load ở train_ml.ipynb
TIME_BINS = 128    # số điểm thời gian sau downsample
BUILD_TEST = True  # build feature test để tạo submission

print(f"Config: LIMIT={LIMIT}, TIME_BINS={TIME_BINS}, BUILD_TEST={BUILD_TEST}")


## 2. Build features từ raw → CSV
Bước chậm nhất — chỉ cần chạy một lần.

In [ ]:
repository = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
pipeline = ArielPreprocessFeaturePipeline(
    PreprocessConfig(target_time_bins=TIME_BINS, apply_cds=True, detrend_degree=2, smooth_window=5),
    FeatureConfig(spectral_bin_sizes=(1, 2, 4, 8, 16, 32, 64), include_per_wavelength_depths=True),
)
builder = ArielDatasetBuilder(repository=repository, pipeline=pipeline)

train_path = PRECOMPUTED_DIR / f"features_train_L{LIMIT}_T{TIME_BINS}.csv"
res = builder.build_feature_csv("train", output_path=train_path, limit=LIMIT,
                                aggregate_observations=True, on_error="skip")
print("Train features:", res.features.shape, "| failures:", len(res.failures), "->", train_path)

if BUILD_TEST:
    test_path = PRECOMPUTED_DIR / f"features_test_T{TIME_BINS}.csv"
    rest = builder.build_feature_csv("test", output_path=test_path, limit=None,
                                     aggregate_observations=True, on_error="skip")
    print("Test features:", rest.features.shape, "| failures:", len(rest.failures), "->", test_path)


## 3. Push lên branch `running`
Sau khi build xong, commit file trong `precomputed/` để `train_ml.ipynb` load lại:
```bash
cd /kaggle/working/ML_IT3190E_Project
git add precomputed/features_*.csv && git commit -m 'Add precomputed ML features' && git push
```
(hoặc Save Version rồi attach output notebook làm dataset cho `train_ml.ipynb`.)